<a href="https://colab.research.google.com/github/tassid/relacoes_fisicas_ambiente/blob/main/codigos_python/trabalho_aula_01.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Trabalho da Aula 01 · Área de estudo do grupo
### Maringá, Londrina e Apucarana (norte do Paraná)

**Disciplina:** Relações Físicas do Ambiente Agrícola (PPGTCA/UTFPR)

| Item pedido | Seção |
|---|---|
| Ambiente configurado, com o projeto de cada integrante | 1 |
| Área de estudo e justificativa agronômica | 2 |
| Mapa interativo com o limite da área | 3 |
| Área total, altitude média e chuva de um ano | 4 |
| Pergunta agroclimática do grupo | 5 |

## 1. Ambiente configurado
___

Cada integrante usa o **próprio projeto** do Google Cloud registrado no Earth Engine. Para rodar, troque `QUEM_ESTA_RODANDO` pelo seu nome; o notebook usa o ID correspondente.

> ⚠️ Se aparecer *"Google Earth Engine API has not been used in project ... or it is disabled"*, a API ainda não foi ativada no projeto. Abra o link que aparece no erro, clique em **Ativar** e rode a célula de novo depois de alguns minutos.

In [ ]:
!pip install -q earthengine-api geemap

In [ ]:
import ee
import geemap
import pandas as pd

# ID do projeto do Google Cloud de cada integrante
PROJETOS = {
    'Tassiane': 'ppgtca-relacoes-fisicas',
    'Eduardo':  'eduproject-510923',
    # 'Nome': 'id-do-projeto',   # demais integrantes
}

QUEM_ESTA_RODANDO = 'Tassiane'
ID_PROJETO = PROJETOS[QUEM_ESTA_RODANDO]

ee.Authenticate()
ee.Initialize(project=ID_PROJETO)
print(ee.String(f'Earth Engine conectado! ({QUEM_ESTA_RODANDO}: {ID_PROJETO})').getInfo())

## 2. Área de estudo
___

A área de estudo é formada por três municípios vizinhos do **norte do Paraná**: **Maringá**, **Londrina** e **Apucarana**. Eles estão no Terceiro Planalto Paranaense, sobre solos derivados de basalto (a "terra roxa"), sob clima Cfa (subtropical úmido, verão quente e sem estação seca definida).

**Justificativa agronômica**

- **Cultura predominante:** a região tem uma das agriculturas de grãos mais intensivas do estado. O sistema típico é **soja no verão** seguida de **milho segunda safra** (safrinha) no outono-inverno, com trigo em parte das áreas. O café, que marcou a ocupação do norte do Paraná, ainda tem presença, principalmente em Apucarana.
- **Importância regional:** Londrina e Maringá são polos do agronegócio paranaense, com cooperativas, cerealistas, indústrias de insumos e centros de pesquisa agropecuária. O que acontece com o clima nesses municípios afeta a produção de toda a região ao redor.
- **Problema climático de interesse:** as duas safras dependem do clima de formas diferentes. A **soja** é sensível a **veranicos** (períodos sem chuva no verão) na floração e no enchimento de grãos, como na estiagem da safra 2021/22. O **milho segunda safra** é semeado no fim do verão e fica exposto à **queda das chuvas no outono** e ao risco de **geada** no inverno, o mesmo risco histórico do café.
- **Por que três municípios:** eles estão a menos de 80 km entre si, mas formam um pequeno gradiente de altitude. Isso permite comparar locais com clima parecido e testar se modelos calibrados em um município valem para os outros.

In [ ]:
MUNICIPIOS = ['Maringá', 'Londrina', 'Apucarana']
NOME_GAUL = {'Maringá': 'Maringa', 'Londrina': 'Londrina', 'Apucarana': 'Apucarana'}   # na base GAUL, sem acento
CORES = {'Maringá': 'red', 'Londrina': 'blue', 'Apucarana': 'orange'}

gaul_pr = ee.FeatureCollection('FAO/GAUL/2015/level2').filter(ee.Filter.eq('ADM1_NAME', 'Parana'))

AREAS = {}
for m in MUNICIPIOS:
    AREAS[m] = gaul_pr.filter(ee.Filter.eq('ADM2_NAME', NOME_GAUL[m]))
    print(f'{m}: {AREAS[m].size().getInfo()} feição encontrada')

# A área de estudo é a união dos três municípios
area_estudo = gaul_pr.filter(ee.Filter.inList('ADM2_NAME', list(NOME_GAUL.values())))
geom_estudo = area_estudo.geometry()

## 3. Mapa interativo
___

Limites dos três municípios sobre o relevo (SRTM) e a chuva do ano escolhido (CHIRPS). Use o controle de camadas, no canto superior direito, para ligar e desligar cada camada.

In [ ]:
ANO_CHUVA = 2024   # ano escolhido para a chuva total

srtm = ee.Image('USGS/SRTMGL1_003')
chuva_ano = (ee.ImageCollection('UCSB-CHG/CHIRPS/DAILY')
             .filterDate(f'{ANO_CHUVA}-01-01', f'{ANO_CHUVA + 1}-01-01')
             .sum())

mapa = geemap.Map()
mapa.add_basemap('HYBRID')
mapa.addLayer(srtm.clip(geom_estudo), {'min': 300, 'max': 900,
              'palette': ['006837', 'a6d96a', 'fee08b', 'd73027']}, 'Altitude (m)', False)
mapa.addLayer(chuva_ano.clip(geom_estudo), {'min': 1000, 'max': 2000,
              'palette': ['f7fbff', '6baed6', '08306b']}, f'Chuva {ANO_CHUVA} (mm)', False)
for m in MUNICIPIOS:
    contorno = ee.Image().paint(AREAS[m], 0, 3)
    mapa.addLayer(contorno, {'palette': [CORES[m]]}, f'Limite de {m}')
mapa.centerObject(area_estudo, 9)
mapa

## 4. Três informações sobre a área
___

Para cada município e para a área inteira:

- **Área total (km²):** área do polígono do município (FAO/GAUL).
- **Altitude média (m):** média do SRTM (30 m) dentro do limite.
- **Chuva total (mm):** soma das imagens diárias do CHIRPS no ano escolhido, média dentro do limite.

In [ ]:
def tres_informacoes(geom):
    area_km2 = geom.area().divide(1e6)
    altitude = srtm.reduceRegion(ee.Reducer.mean(), geom, scale=30, maxPixels=1e10).get('elevation')
    chuva = chuva_ano.reduceRegion(ee.Reducer.mean(), geom, scale=5566, maxPixels=1e10).get('precipitation')
    return ee.Dictionary({'area': area_km2, 'altitude': altitude, 'chuva': chuva}).getInfo()


linhas = []
for nome, geom in [(m, AREAS[m].geometry()) for m in MUNICIPIOS] + [('Área de estudo (total)', geom_estudo)]:
    info = tres_informacoes(geom)
    linhas.append({'Município': nome,
                   'Área total (km²)': round(info['area'], 1),
                   'Altitude média (m)': round(info['altitude'], 0),
                   f'Chuva total {ANO_CHUVA} (mm)': round(info['chuva'], 0)})

tabela = pd.DataFrame(linhas).set_index('Município')
tabela

> 💬 **Para discutir:** o CHIRPS combina satélite e estações, com pixel de ~5,5 km. A chuva de um ano isolado pode ficar bem acima ou abaixo da média; comparem com a estação do INMET de cada cidade.

## 5. Pergunta agroclimática do grupo
___

> **No norte do Paraná, é possível estimar com precisão suficiente a demanda hídrica das culturas usando apenas a temperatura do ar e, com isso, explicar quanto da variação da produtividade da soja e do milho segunda safra se deve ao déficit hídrico?**

Como a pergunta se encaixa nos projetos da disciplina:

| Projeto | Parte da pergunta |
|---|---|
| P1 · Elementos meteorológicos | Estimar a radiação solar e a ETo só com temperatura, e medir o erro dessa estimativa |
| P2 · Chuva, ETo e balanço hídrico | Usar essa ETo no balanço hídrico e identificar os períodos de deficiência nas duas safras |
| P3 · Produtividade | Penalizar a produtividade potencial pelo déficit hídrico (MZA-FAO) e comparar com a produtividade do IBGE |